# grid-rbd kinematics — FK, frame Jacobians & operational-space inertia

The **kinematic surface** of `grid_rbd`, all batched over axis 0:

- `end_effector_pose(q)` and the large-batch `fk_batched(q)`
- `end_effector_pose_gradient(q)` / `end_effector_pose_hessian(q)` (tangent-space, pinocchio convention)
- `frame_jacobian(q, target_jid=, reference_frame=)` and `frame_jacobian_dot(q, qd, ...)` — **runtime** frame selection
- centroidal kinematics `com(q)`, `ccrba(q, qd)`, and `osc_inertia(q)`

Every section ends in an `assert` that cross-checks GRiD against a **finite-difference** of its own forward map — so a green Run All validates the numbers, not just 'no exception'. (Finite-diff is hermetic: no second library or frame-name plumbing needed.)

**Setup:** a CUDA GPU + `nvcc` on PATH, and `grid_rbd` installed editable from this repo — `pip install -e bindings/` from the repo root (see [`notebooks/README.md`](README.md) for backends + the import-resolution check). iiwa14 (7-DOF fixed-base) is the cheapest robot — first `register_robot` is seconds-to-tens-of-seconds of `nvcc`; a re-run is a cache hit.

In [ ]:
import numpy as np
from pathlib import Path
import grid_rbd

URDF = next(p / 'robot_assets' / 'iiwa14.urdf' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'robot_assets' / 'iiwa14.urdf').exists())
assert URDF.exists(), URDF
np.random.seed(0)

## 1. Register the robot (one-time; cache-hit on re-run)

In [ ]:
h = grid_rbd.register_robot('iiwa14_kin', urdf_path=str(URDF),
                            floating_base=False, max_batch_size=64)
NJ, NV, NEE = h.num_joints, h.num_vel, h.num_ees
print(h)
print(f'num_joints={NJ}  num_vel={NV}  num_ees={NEE}  num_bodies={h.num_bodies}')

## 2. Forward kinematics — `end_effector_pose` and `fk_batched`

`end_effector_pose(q)` returns `[xyz, rpy]` per EE, shape `(B, 6*NEE)`. `fk_batched(q)` is the large-batch one-block-per-sample variant and returns `[tx,ty,tz, qw,qx,qy,qz]` (position + unit quaternion) for the leaf frame; `use_warp=True` runs the warp-cooperative inner and returns identical poses.

In [ ]:
B = 8
q = np.random.randn(B, NJ).astype(np.float32) * 0.4

pose = h.end_effector_pose(q)              # (B, 6*NEE) = [xyz, rpy]
pose7 = h.fk_batched(q)                    # (B, 7)     = [xyz, quat(wxyz)]
pose7_warp = h.fk_batched(q, use_warp=True)
print('end_effector_pose:', pose.shape, '  fk_batched:', pose7.shape)

# The two FK surfaces agree on translation (xyz).
xyz_pose = pose.reshape(B, NEE, 6)[:, 0, :3]
xyz_fk   = pose7[:, :3]
assert np.allclose(xyz_pose, xyz_fk, atol=1e-4), np.abs(xyz_pose - xyz_fk).max()
# thread vs warp variant are bit-for-bit the same pose.
assert np.allclose(pose7, pose7_warp, atol=1e-5)
print('FK surfaces agree (pose vs fk_batched, thread vs warp).')

## 3. End-effector pose Jacobian & Hessian

`end_effector_pose_gradient(q)` is `d(pose)/dv`, shape `(B, 6*NEE, NV)` (tangent-space / pinocchio convention). We validate its **translation rows** against a central finite-difference of `end_effector_pose` — for fixed-base the tangent basis is the joint basis, so the xyz rows match `d(xyz)/dq` directly. `end_effector_pose_hessian(q)` returns `(B, 6*NEE, NV, NV)`.

In [ ]:
q0 = (np.random.randn(1, NJ).astype(np.float32) * 0.3)
J = h.end_effector_pose_gradient(q0)        # (1, 6*NEE, NV)
Hess = h.end_effector_pose_hessian(q0)      # (1, 6*NEE, NV, NV)
print('ee_pose_gradient:', J.shape, '  ee_pose_hessian:', Hess.shape)

# Central finite-difference of the xyz pose rows wrt q.
eps = 1e-3
fd = np.zeros((3, NJ), dtype=np.float64)
for j in range(NJ):
    dq = np.zeros((1, NJ), dtype=np.float32); dq[0, j] = eps
    pp = h.end_effector_pose(q0 + dq).reshape(1, NEE, 6)[0, 0, :3]
    pm = h.end_effector_pose(q0 - dq).reshape(1, NEE, 6)[0, 0, :3]
    fd[:, j] = (pp.astype(np.float64) - pm.astype(np.float64)) / (2 * eps)

J_xyz = J[0, :3, :NJ].astype(np.float64)    # translation rows, joint cols
err = np.abs(J_xyz - fd).max()
print(f'analytic ee-Jacobian vs finite-diff: max|err| = {err:.2e}')
assert err < 2e-2, err

## 4. Geometric frame Jacobian — **runtime** frame & target selection

`frame_jacobian(q, target_jid=, reference_frame=)` returns the geometric `6 x NV` Jacobian `[linear; angular]`. Both arguments are now **runtime** parameters of the GPU surface (no recompile to switch frames):

- `reference_frame`: `'LOCAL'`, `'WORLD'`, or `'LOCAL_WORLD_ALIGNED'` (default).
- `target_jid`: the frame's joint id (default: the leaf EE joint baked at codegen).

The three reference frames give genuinely different Jacobians; the `LOCAL_WORLD_ALIGNED` linear block equals the translation Jacobian we finite-differenced above.

In [ ]:
Jlwa = h.frame_jacobian(q0, reference_frame='LOCAL_WORLD_ALIGNED')  # (1,6,NV)
Jloc = h.frame_jacobian(q0, reference_frame='LOCAL')
Jwld = h.frame_jacobian(q0, reference_frame='WORLD')
print('frame_jacobian:', Jlwa.shape)

# The three frames are distinct rotations of the same geometric object.
assert not np.allclose(Jlwa, Jloc, atol=1e-3), 'LOCAL should differ from LWA'
assert not np.allclose(Jlwa, Jwld, atol=1e-3), 'WORLD should differ from LWA'

# LWA linear block (rows 0:3) == d(xyz)/dq from section 3 (same FD target).
err_lin = np.abs(Jlwa[0, :3, :NJ].astype(np.float64) - fd).max()
print(f'LWA linear block vs xyz finite-diff: max|err| = {err_lin:.2e}')
assert err_lin < 2e-2, err_lin

# Targeting an intermediate joint selects a DIFFERENT frame, so its
# Jacobian differs from the leaf-EE default (only the chain up to the target
# frame contributes to its motion).
mid = NJ // 2
Jdefault = h.frame_jacobian(q0)               # leaf-EE frame (baked default)
Jmid = h.frame_jacobian(q0, target_jid=mid)   # an intermediate frame
diff = np.abs(Jdefault[0] - Jmid[0]).max()
print(f'target_jid={mid} vs leaf default: max|ΔJ| = {diff:.2e}')
assert diff > 1e-3, 'target_jid had no effect'

## 5. Frame Jacobian time-derivative `Jdot`

`frame_jacobian_dot(q, qd, ...)` is `dJ/dt` along `v = qd`, same `(B, 6, NV)` shape and same runtime `target_jid`/`reference_frame`. We validate against a finite-difference of `frame_jacobian` along the path `q(t) = q + t·qd`.

In [ ]:
qd0 = (np.random.randn(1, NJ).astype(np.float32) * 0.5)
Jdot = h.frame_jacobian_dot(q0, qd0)        # (1, 6, NV)

dt = 1e-3
Jp = h.frame_jacobian(q0 + dt * qd0).astype(np.float64)
Jm = h.frame_jacobian(q0 - dt * qd0).astype(np.float64)
Jdot_fd = (Jp - Jm) / (2 * dt)
err = np.abs(Jdot[0].astype(np.float64) - Jdot_fd[0]).max()
print(f'frame_jacobian_dot vs finite-diff along qd: max|err| = {err:.2e}')
assert err < 5e-2, err

## 6. Centroidal kinematics & operational-space inertia

- `com(q)` → `(p_com (B,3), J_com (B,3,NV))` — CoM world position and its Jacobian.
- `ccrba(q, qd)` → `(A (B,6,NV), h (B,6))` — centroidal momentum matrix and momentum `h = A·qd`.
- `osc_inertia(q)` → `(B,6,6)` task-space inertia `Λ = (J·M⁻¹·Jᵀ)⁻¹` for the leaf EE.

We validate `J_com` against a finite-difference of `p_com`, and `h = A·qd` by construction.

In [ ]:
p_com, J_com = h.com(q0)
A, hmom = h.ccrba(q0, qd0)
Lambda = h.osc_inertia(q0)
print('p_com:', p_com.shape, ' J_com:', J_com.shape,
      ' A:', A.shape, ' h:', hmom.shape, ' Lambda:', Lambda.shape)

# J_com vs central finite-diff of p_com.
fd_com = np.zeros((3, NJ), dtype=np.float64)
for j in range(NJ):
    dq = np.zeros((1, NJ), dtype=np.float32); dq[0, j] = eps
    fd_com[:, j] = (h.com(q0 + dq)[0][0].astype(np.float64)
                    - h.com(q0 - dq)[0][0].astype(np.float64)) / (2 * eps)
err_com = np.abs(J_com[0, :, :NJ].astype(np.float64) - fd_com).max()
print(f'J_com vs finite-diff: max|err| = {err_com:.2e}')
assert err_com < 2e-2, err_com

# Centroidal momentum identity h == A @ qd.
h_from_A = (A[0].astype(np.float64) @ qd0[0].astype(np.float64))
assert np.allclose(h_from_A, hmom[0].astype(np.float64), atol=1e-3), \
    np.abs(h_from_A - hmom[0]).max()

# Lambda is symmetric positive-definite (task inertia).
L = Lambda[0].astype(np.float64)
assert np.allclose(L, L.T, atol=1e-3), 'Lambda not symmetric'
assert np.all(np.linalg.eigvalsh(0.5 * (L + L.T)) > 0), 'Lambda not PD'
print('h == A@qd and Lambda SPD: centroidal + OSC surface validated.')

All kinematic surfaces (`end_effector_pose` / `fk_batched`, `end_effector_pose_gradient` / `_hessian`, `frame_jacobian` / `_dot` with runtime frame+target, `com` / `ccrba` / `osc_inertia`) cross-checked against finite-difference of their own forward maps. Green ⇒ numbers validated.